# Xử lý dữ liệu CV - CuCoMOF Glucose Sensor
Dataset gồm các đường cong Cyclic Voltammetry (CV) đo cường độ dòng điện (µA) theo điện thế (V) tại các nồng độ glucose khác nhau.

## 1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Đọc dữ liệu thô

File CSV dùng tab (`\t`) làm separator, có 2 dòng header và một số dòng có tab kép (dữ liệu bị lệch cột). Cần đọc thủ công để xử lý.

In [ ]:
concentrations = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]  # mM

data_rows = []
with open("CUCOMOF.csv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        # Bỏ dòng trống và 3 dòng header đầu
        if not line or i in [0, 1, 2]:
            continue
        parts = [p.strip() for p in line.split('\t')]
        # Lọc bỏ chuỗi rỗng do tab kép
        clean = [p for p in parts if p != '']
        if len(clean) >= 11:
            clean = clean[:11]
        else:
            clean += [np.nan] * (11 - len(clean))
        data_rows.append(clean)

df_raw = pd.DataFrame(data_rows, columns=['V'] + [f'{c}mM' for c in concentrations])
df_raw = df_raw.apply(pd.to_numeric, errors='coerce')

print(f"Kích thước dữ liệu thô: {df_raw.shape}")
print(f"V range: {df_raw['V'].min():.3f} → {df_raw['V'].max():.3f} V")
df_raw.head()

## 3. Lọc chiều quét đi (-0.2V → 0.8V)

Dữ liệu CV gồm 2 chiều: đi (-0.2 → 0.8V) và về (0.8 → -0.2V). Chỉ lấy chiều đi (100 điểm đầu tiên đến V_max).

In [ ]:
v_max_idx = df_raw['V'].idxmax()
print(f"V max tại index {v_max_idx}: {df_raw['V'][v_max_idx]:.3f} V")

df_forward = df_raw.iloc[:v_max_idx + 1].copy().reset_index(drop=True)
print(f"Chiều đi: {len(df_forward)} điểm")

## 4. Làm tròn điện thế về step 0.1V

Chọn các mốc điện thế: -0.2, -0.1, 0.0, 0.1, ..., 0.8V (tổng 11 mốc).

In [ ]:
target_V = [round(v, 1) for v in np.arange(-0.2, 0.9, 0.1)]

df_forward['V_rounded'] = df_forward['V'].round(1)
df_filtered = df_forward[df_forward['V_rounded'].isin(target_V)].copy()

# Nếu có nhiều hàng cùng V_rounded, lấy hàng gần nhất với mốc
df_filtered['V_diff'] = abs(df_filtered['V'] - df_filtered['V_rounded'])
df_filtered = (df_filtered
               .sort_values('V_diff')
               .drop_duplicates('V_rounded')
               .sort_values('V_rounded')
               .drop(columns=['V', 'V_diff'])
               .rename(columns={'V_rounded': 'V'})
               .reset_index(drop=True))

print(f"Số mốc điện thế: {len(df_filtered)}")
print(f"V values: {df_filtered['V'].tolist()}")
df_filtered

## 5. Xoay ngang (Transpose)

Chuyển từ dạng `(V × nồng_độ)` sang dạng ML chuẩn: mỗi **hàng = 1 mẫu** (1 nồng độ glucose), mỗi **cột = 1 feature** (cường độ dòng tại điện thế V).

In [ ]:
df_final = df_filtered.set_index('V').T.reset_index()
df_final.columns = ['glucose_mM'] + [f'V_{v:.1f}' for v in df_filtered['V'].tolist()]
df_final['glucose_mM'] = df_final['glucose_mM'].str.replace('mM', '').astype(float)

print(f"Số mẫu   : {len(df_final)}")
print(f"Số features: {len(df_final.columns) - 1} (cường độ dòng tại 11 mốc V)")
print(f"Cột y (target): glucose_mM")
df_final

## 6. Visualize dữ liệu

In [ ]:
feature_cols = [c for c in df_final.columns if c.startswith('V_')]
v_vals = [float(c.replace('V_', '')) for c in feature_cols]

plt.figure(figsize=(10, 6))
for _, row in df_final.iterrows():
    plt.plot(v_vals, row[feature_cols].values, marker='o', markersize=4,
             label=f"{row['glucose_mM']} mM")

plt.xlabel('Điện thế (V)')
plt.ylabel('Cường độ dòng (µA)')
plt.title('CV curves theo nồng độ Glucose')
plt.legend(title='Glucose', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Lưu file

In [ ]:
df_final.to_csv("CUCOMOF_processed.csv", index=False)
print("✅ Đã lưu: CUCOMOF_processed.csv")